### SCD Type 2 — Step 1: Create the dimension

In [0]:
from pyspark.sql.functions import (
    col, current_date, lit,
    to_date
)

# Read clean customer data
df_customers = spark.table("silver_customers")

# Create initial SCD Type 2 customer dimension
df_customer_dim = (
    df_customers
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

display(df_customer_dim)

customer_id,name,city,state,registration_date,effective_start_date,effective_end_date,is_current
C0008,Yahvi Ratta,Delhi,Delhi,2025-07-02,2026-10-02,null,true
C0015,Samuel Kara,Pune,Maharashtra,2025-05-28,2026-10-02,null,true
C0038,Qasim Sen,Mumbai,Maharashtra,2025-06-19,2026-10-02,null,true
C0045,Hemang Zacharia,Kochi,Kerala,2025-04-29,2026-10-02,null,true
C0050,Veda Wagle,Mumbai,Maharashtra,2024-01-27,2026-10-02,null,true
C0065,Maanav Goda,Mumbai,Maharashtra,2024-10-14,2026-10-02,null,true
C0086,Wakeeta Khosla,Pune,Maharashtra,2024-03-02,2026-10-02,null,true
C0095,Falan Choudhry,Hyderabad,Telangana,2026-06-30,2026-10-02,null,true
C0136,Jagdish Ravi,Mumbai,Maharashtra,2024-07-05,2026-10-02,null,true
C0198,Vincent Kapoor,Pune,Maharashtra,2026-05-16,2026-10-02,null,true


#### Step 2 — Create the SCD2 table and Vderify

In [0]:
df_customer_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer_scd2")

display(spark.table("dim_customer_scd2"))

customer_id,name,city,state,registration_date,effective_start_date,effective_end_date,is_current
C0008,Yahvi Ratta,Delhi,Delhi,2025-07-02,2026-10-02,null,true
C0015,Samuel Kara,Pune,Maharashtra,2025-05-28,2026-10-02,null,true
C0038,Qasim Sen,Mumbai,Maharashtra,2025-06-19,2026-10-02,null,true
C0045,Hemang Zacharia,Kochi,Kerala,2025-04-29,2026-10-02,null,true
C0050,Veda Wagle,Mumbai,Maharashtra,2024-01-27,2026-10-02,null,true
C0065,Maanav Goda,Mumbai,Maharashtra,2024-10-14,2026-10-02,null,true
C0086,Wakeeta Khosla,Pune,Maharashtra,2024-03-02,2026-10-02,null,true
C0095,Falan Choudhry,Hyderabad,Telangana,2026-06-30,2026-10-02,null,true
C0136,Jagdish Ravi,Mumbai,Maharashtra,2024-07-05,2026-10-02,null,true
C0198,Vincent Kapoor,Pune,Maharashtra,2026-05-16,2026-10-02,null,true


#### Step 4 — Create a simulated customer update

In [0]:
from pyspark.sql.functions import lit

# Pick one existing customer
customer_id_to_update = (
    spark.table("silver_customers")
    .select("customer_id")
    .first()["customer_id"]
)

# Create an updated version of that customer
df_customer_update = (
    spark.table("silver_customers")
    .filter(col("customer_id") == customer_id_to_update)
    .withColumn("city", lit("Bangalore"))
)

display(df_customer_update)

customer_id,name,city,state,registration_date
C0008,Yahvi Ratta,Bangalore,Delhi,2025-07-02


#### Step 5 — Close the old version

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date

# Access the SCD2 Delta table
customer_dim = DeltaTable.forName(spark, "dim_customer_scd2")

# Update the existing current record
(
    customer_dim.alias("target")
    .merge(
        df_customer_update.alias("source"),
        "target.customer_id = source.customer_id AND target.is_current = true"
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date": current_date(),
            "is_current": lit(False)
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

#### Step 6 — Insert the new version

In [0]:
from pyspark.sql.functions import current_date, lit

df_new_customer_version = (
    df_customer_update
    .withColumn("effective_start_date", current_date())
    .withColumn("effective_end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

df_new_customer_version.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dim_customer_scd2")

#### Step 7 — Verify the history

In [0]:
display(
    spark.table("dim_customer_scd2")
    .filter(col("customer_id") == customer_id_to_update)
    .orderBy("effective_start_date")
)

customer_id,name,city,state,registration_date,effective_start_date,effective_end_date,is_current
C0008,Yahvi Ratta,Delhi,Delhi,2025-07-02,2026-10-02,2026-10-02,false
C0008,Yahvi Ratta,Bangalore,Delhi,2025-07-02,2026-10-02,null,true
